# 12a — Rating Mirror prep (5,000-user cohort)

Build dual-channel tensors for **Rating Mirror** on the cohort from notebook **09b**:

- **Channel A1 (real):** user rating one-hot — identical to `channel1` from 09b (same as Rating Personality B1)
- **Channel A2 (imaginary):** rating complement one-hot — index `k2 = 9 - k1` so `RATING_LEVELS[k1] + RATING_LEVELS[k2] = 5.5`

| Part | Section |
|------|--------|
| Part 0 | Setup |
| Part 1 | Load 09b outputs |
| Part 2 | Build Rating Mirror channel 2 |
| Part 3 | Save |
| Part 4 | Verification |

**Prerequisites:** notebook 09b (`data/processed/` artifacts).

## Part 0 — Setup

In [1]:
from pathlib import Path

import numpy as np

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

out_dir = root / "data" / "processed"
assert out_dir.exists(), f"Missing {out_dir} — run notebook 09b first."

RATING_LEVELS = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0], dtype=np.float64)
K = len(RATING_LEVELS)
rating_to_idx = {float(r): i for i, r in enumerate(RATING_LEVELS)}

print(f"Project root: {root}")
print(f"Processed:    {out_dir}")
print(f"K={K}, RATING_LEVELS={RATING_LEVELS.tolist()}")

Project root: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys
Processed:    /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed
K=10, RATING_LEVELS=[0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]


## Part 1 — Load 09b outputs

In [2]:
path_cohort = out_dir / "cohort_user_ids.npy"
path_vocab = out_dir / "movie_vocab.npy"
path_mask = out_dir / "mask.npy"
path_ch1 = out_dir / "channel1_softmax.npy"

for p in (path_cohort, path_vocab, path_mask, path_ch1):
    assert p.exists(), f"Missing {p} — run notebook 09b first."

cohort_user_ids = np.load(path_cohort)
movie_vocab = np.load(path_vocab)
mask = np.load(path_mask, mmap_mode="r")
channel1 = np.load(path_ch1, mmap_mode="r")

n_users = len(cohort_user_ids)
n_movies = len(movie_vocab)
user_to_row = {int(uid): i for i, uid in enumerate(cohort_user_ids)}
movie_to_col = {int(mid): j for j, mid in enumerate(movie_vocab)}

COMPLEMENT = {float(r): 5.5 - float(r) for r in RATING_LEVELS}

print(f"cohort_user_ids: {cohort_user_ids.shape}  {cohort_user_ids.dtype}")
print(f"movie_vocab:     {movie_vocab.shape}  {movie_vocab.dtype}")
print(f"mask:            {mask.shape}  {mask.dtype}")
print(f"channel1:        {channel1.shape}  {channel1.dtype}")
print(f"COMPLEMENT: {COMPLEMENT}")
assert channel1.shape == (n_users, n_movies, K)
assert mask.shape == (n_users, n_movies)

cohort_user_ids: (5000,)  int64
movie_vocab:     (13129,)  int64
mask:            (5000, 13129)  int8
channel1:        (5000, 13129, 10)  float32
COMPLEMENT: {0.5: 5.0, 1.0: 4.5, 1.5: 4.0, 2.0: 3.5, 2.5: 3.0, 3.0: 2.5, 3.5: 2.0, 4.0: 1.5, 4.5: 1.0, 5.0: 0.5}


## Part 2 — Build Rating Mirror channel 2

`channelA1 = channel1` (identical one-hot user ratings).  
`channelA2[i, j, :]` is the complement one-hot: if `channel1` has a 1 at index `k1`, place a 1 at `k2 = 9 - k1`.  
Unrated positions (`mask == 0`) stay all-zero.  

Implementation: flip the last axis of `channel1` (equivalent to `k2 = 9 - k1`), written via memmap to avoid holding two full copies in RAM.

In [3]:
channelA1 = channel1  # alias; identical to channel B1 / channel1

path_a2 = out_dir / "channelA2_mirror.npy"
channelA2 = np.lib.format.open_memmap(
    path_a2,
    mode="w+",
    dtype=np.float32,
    shape=(n_users, n_movies, K),
)

CHUNK = 250
print(f"Building channelA2 (mirror) → {path_a2.name}  chunks of {CHUNK} users …")
for i0 in range(0, n_users, CHUNK):
    i1 = min(i0 + CHUNK, n_users)
    # Reverse along K: one-hot at k1 → one-hot at (K-1-k1) = 9-k1
    channelA2[i0:i1] = np.flip(np.asarray(channel1[i0:i1], dtype=np.float32), axis=-1)
    if (i0 // CHUNK) % 4 == 0 or i1 == n_users:
        print(f"  users {i0:,}–{i1 - 1:,}")

channelA2.flush()
print(f"channelA1: alias of channel1  {channelA1.shape}")
print(f"channelA2: {channelA2.shape}  {channelA2.dtype}")

Building channelA2 (mirror) → channelA2_mirror.npy  chunks of 250 users …
  users 0–249
  users 1,000–1,249
  users 2,000–2,249
  users 3,000–3,249
  users 4,000–4,249
  users 4,750–4,999
channelA1: alias of channel1  (5000, 13129, 10)
channelA2: (5000, 13129, 10)  float32


## Part 3 — Save

`channelA2_mirror.npy` is written in Part 2.  
`channelA1` is identical to `channel1` — no re-save.

In [4]:
print(f"Saved: {path_a2}  shape={channelA2.shape}  dtype={channelA2.dtype}")
print(f"Skipped channelA1 re-save (identical to {path_ch1.name})")

Saved: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/channelA2_mirror.npy  shape=(5000, 13129, 10)  dtype=float32
Skipped channelA1 re-save (identical to channel1_softmax.npy)


## Part 4 — Verification

In [5]:
# Chunked checks — avoid loading two full (5000, 13129, 10) tensors into RAM
CHUNK = 250
nz1 = nz2 = 0
n_obs = 0
sym_ok = True

for i0 in range(0, n_users, CHUNK):
    i1 = min(i0 + CHUNK, n_users)
    c1 = np.asarray(channel1[i0:i1], dtype=np.float32)
    c2 = np.asarray(channelA2[i0:i1], dtype=np.float32)
    m = np.asarray(mask[i0:i1])

    nz1 += int((c1.sum(axis=-1) > 0).sum())
    nz2 += int((c2.sum(axis=-1) > 0).sum())

    obs_i, obs_j = np.where(m == 1)
    if len(obs_i) == 0:
        continue
    k1 = c1[obs_i, obs_j].argmax(axis=-1)
    k2 = c2[obs_i, obs_j].argmax(axis=-1)
    if not np.all(k1 + k2 == 9):
        sym_ok = False
        break
    n_obs += len(obs_i)

assert nz1 == nz2, f"nonzero columns: channel1={nz1}, channelA2={nz2}"
print(f"✓ nonzero observed positions: {nz1:,}")
assert sym_ok, "complement symmetry failed"
print(f"✓ complement symmetry: argmax(ch1)+argmax(chA2)==9 for all {n_obs:,} observed positions")

# Example: user row 0, one observed movie
i_ex = 0
js = np.where(np.asarray(mask[i_ex]) == 1)[0]
assert len(js) > 0
j_ex = int(js[0])
uid = int(cohort_user_ids[i_ex])
mid = int(movie_vocab[j_ex])
oh1 = np.asarray(channel1[i_ex, j_ex])
oh2 = np.asarray(channelA2[i_ex, j_ex])
k1_ex = int(oh1.argmax())
k2_ex = int(oh2.argmax())
r = float(RATING_LEVELS[k1_ex])
r_comp = float(RATING_LEVELS[k2_ex])

print(f"\nExample — userId={uid} (row {i_ex}), movieId={mid} (col {j_ex})")
print(f"  rating={r}, complement={r_comp}  (COMPLEMENT[{r}]={COMPLEMENT[r]})")
print(f"  channel1  one-hot (k={k1_ex}): {oh1.tolist()}")
print(f"  channelA2 one-hot (k={k2_ex}): {oh2.tolist()}")
assert abs(r + r_comp - 5.5) < 1e-9
assert k1_ex + k2_ex == 9

print("\nAll checks passed.")


✓ nonzero observed positions: 4,119,189
✓ complement symmetry: argmax(ch1)+argmax(chA2)==9 for all 4,119,189 observed positions

Example — userId=118205 (row 0), movieId=1 (col 0)
  rating=4.0, complement=1.5  (COMPLEMENT[4.0]=1.5)
  channel1  one-hot (k=7): [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0]
  channelA2 one-hot (k=2): [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

All checks passed.
